# 02 Data Pattern Analysis — Automated（整合版）

**Mirrors the CONFIG/engine structure of `01_data_exploration_auto.py`.**

- **CONFIG（要改的東西）**: pattern 清單（要偵測什麼）+ 每個資料集的欄位名稱對應
- **Engine（不用改）**: 掃描 + 統計 + 印範例，**不做清理決策**
- **輸出**: 統計報表（給人看）→ 人工判斷後手動填進 `CLEANING_RULES`（給 03 用）

## 整體架構：四塊怎麼互相呼叫（先看大局，再看 Step 細節）

跟 01 最大的不同：**01 只有一個「體檢站」（`run_exploration`），機器自己下結論**；**02 是「四塊」，多了一塊「人工判斷」，因為 pattern 分類這件事機器只能算數字、不能自己決定該不該清**。

```
① CONFIG                    ② Engine（4 個通用 function）        ③ Step 1-5（呼叫②，印報表）         ④ Step 6（人工判斷）
   PATTERNS_ARTIST/SONG  --> run_pattern_scan()              --> Step 1：6 張基礎統計表          --> CLEANING_RULES
   DATASETS 欄位對應           show_examples()                    Step 2：& / 逗號分類                 人親自讀完 ①-③
                              classify_symbol_usage()            Step 3：括號內容分類                  全部報表後，
                              classify_extracted_content()       Step 4：破折號內容分類                手寫判斷結果
                              （只算「有多少、屬於哪一類」，        Step 5：D1 vs D2 vs D3 比較
                               不判斷「該不該清」）
```

**對應到「白板模型」**：
- **① CONFIG** 準備好「要偵測什麼」跟「欄位名稱對照表」
- **② Engine** 是一組通用工具（量尺），本身不認識 D1/D2/D3，只認識「給我一欄資料 + 一份規則，我幫你算」
- **③ Step 1-5** 才是真正把 D1/D2/D3 的資料送進②量尺去量，每量完一項就印一張報表——**但報表本身不會告訴你「該清掉哪些」**
- **④ Step 6** 是妳自己讀完③所有報表後，用人腦判斷寫出來的最終規則——這格程式碼看起來也是「跑出來的」，但裡面的內容（`CLEANING_RULES` 這個 dict）其實是手打的判斷結果，不是機器算出來的

**這就是為什麼 02 的 6 步驟藍圖表要多一欄「誰下結論」**：01 每步都是「機器」，02 除了 Step 6 都是「機器算，人判斷」——這個差異本身就是 02 存在的意義：把人工判斷的過程結構化、留下紀錄，而不是判斷完就消失。

## 02 在幹嘛？—— 6 步驟藍圖

跟 01 最大的不同：**01 每一步機器都能自己下結論**（[OK]/[WARN]）；**02 大部分步驟機器只能「算出來」，結論要人看報表才能下**——這也是為什麼最後多一個 Step 6，是唯一「不是機器跑出來、而是人工寫進去」的步驟。

| Step | 分析項目（對應 function） | 問的問題 | 誰下結論 |
|------|------|------|------|
| **Step 1** | 基礎 Pattern 統計（`run_pattern_scan`） | 這個欄位裡，`&`、`feat.`、括號…這些特殊符號各出現幾筆、佔多少 %？ | 機器（純計數，客觀事實）|
| **Step 2** | `&` / 逗號分類（`classify_symbol_usage`） | 這個符號旁邊有沒有 featuring/feat./with 這種合作標記？還是同一列出現兩次以上？ | 機器分類 → **人看報表判斷**該不該清 |
| **Step 3** | 括號內容分類（`classify_extracted_content` + `PAREN_CATEGORIES`） | 括號裡裝的是 Remix/Live/Remaster 這種版本標記，還是歌名/團名本身的一部分？ | 機器分類 → **人看報表判斷** |
| **Step 4** | 破折號內容分類（同一個 function，換一份 `DASH_CATEGORIES`） | 破折號後面接的是版本標記，還是歌曲副標題？ | 機器分類 → **人看報表判斷** |
| **Step 5** | 跨資料集比較（`three_way` 表） | D1 vs D2 vs D3，同一種模式的出現比例差多少？哪個資料集特別「髒」？ | 機器算出來 → **人看趨勢** |
| **Step 6** | `CLEANING_RULES`（最終交付物） | 綜合以上所有報表，到底哪些 pattern 該清、哪些該留？ | **純人工判斷**（唯一不是機器跑出來的東西）|

> **範圍提醒**：R 版本只對 **D1** 做了 Step 2-4 的深度分類（D2/D3 R 版本只寫了 Step 1 的統計數字 + 用文字寫結論，沒有寫分類程式碼），這裡照實移植，沒有超做。Step 6 的 `CLEANING_RULES` 則涵蓋全部三個資料集，因為那是把 R 原本手寫的文字結論轉成結構化 config，不是新增分析。

In [1]:
import pandas as pd

# Load validated datasets from Stage 01
d1_billboard   = pd.read_pickle('../Data/cleaned_D1.pkl')
d2_spotify_all = pd.read_pickle('../Data/cleaned_D2.pkl')
d3_music       = pd.read_pickle('../Data/cleaned_D3.pkl')

print('Datasets loaded')
print(f'D1 (Billboard): {len(d1_billboard):,} rows')
print(f'D2 (Spotify):   {len(d2_spotify_all):,} rows')
print(f'D3 (Music):     {len(d3_music):,} rows')

Datasets loaded
D1 (Billboard): 330,087 rows
D2 (Spotify):   41,106 rows
D3 (Music):     28,372 rows


---
## ① CONFIG — Only edit this section when adding datasets or patterns

**設計邏輯**：`PATTERNS_ARTIST` / `PATTERNS_SONG` 這兩份清單**只寫一次**，因為 D1/D2/D3 要偵測的模式是一樣的（`&`、`feat.`、括號…）。

真正因資料集而異的只有**欄位名稱**（D1: `artist`/`song`，D2: `artist`/`track`，D3: `artist_name`/`track_name`），所以欄位名稱差異收進 `DATASETS` 這個 dict，跟 pattern 清單分開放。

這樣以後如果想多偵測一種 pattern（比如中文字元），只要改 `PATTERNS_ARTIST` 一個地方，三個資料集會自動一起套用——不用三個地方各改一次。

In [2]:
# ============================================================
# CONFIG -- Only edit this section when adding datasets/patterns 
# ============================================================

# 底層程式碼設定了 case=False 所以無論你在 Config 裡寫大寫、小寫還是混在一起，機器在執行時都會自動把所有變體通通抓出來。
# Patterns shared across ALL datasets' artist/artist_name column
PATTERNS_ARTIST = [
    ("Contains 'Featuring'",       r'Featuring'),
    ("Contains 'feat.' or 'ft.'",  r'feat\.|ft\.'),
    ("Contains '&'",               r'&'),
    ("Contains ','",               r','),
    ("Contains '()' or '[]'",      r'[\(\)\[\]]'), #( 或 ) 或 [ 或 ]
    ("Contains '/'",               r'/'),
    ("Contains 'x' (Collab)",      r'\sx\s'),      #找出前後都有空格的 X
    ("Contains '+' (Collab)",      r'\s\+\s'),     #找出前後都有空格的 +
]

# Patterns shared across ALL datasets' song/track/track_name column
PATTERNS_SONG = [
    ("Contains '()'",           r'\(.*\)'),       #幫我抓出任何被『圓括號 ( )』包起來的完整內容
    ("Contains '[]'",           r'\[.*\]'),       #幫我抓出任何被『方括號 ( )』包起來的完整內容
    ("Contains ' - ' (dash)",   r' - '),
    ("Contains 'feat.'",        r'feat\.'),
    ("Contains 'Remaster'",     r'Remaster'),
    ("Contains 'Live'",         r'Live'),
    ("Contains 'Remix'",        r'Remix'),
    ("Contains 'Radio Edit'",   r'Radio Edit'),
]

# Column-name mapping per dataset -- this is the ONLY thing that
# actually differs between D1 / D2 / D3
DATASETS = {
    "D1": {"df": d1_billboard,   "artist_col": "artist",      "song_col": "song"},
    "D2": {"df": d2_spotify_all, "artist_col": "artist",      "song_col": "track"},
    "D3": {"df": d3_music,       "artist_col": "artist_name", "song_col": "track_name"},
}

---
## ② Engine — No edits needed below this line

`run_pattern_scan()` 吃「一個 df + 一個欄位名稱 + 一份 pattern 清單」，跑完回傳一張統計表（筆數 + 百分比）。

**注意它「不做」什麼**：它不會判斷「該不該清」，只負責回答「有多少」。這是刻意的——判斷留給你自己看報表之後做。

In [3]:
def run_pattern_scan(df, col, patterns):
    """Scan one column against a list of (name, regex) patterns.
    Returns a stats DataFrame -- does NOT decide what to clean."""

    total = len(df)
    rows = []
    for pattern_name, regex in patterns:
        count = df[col].str.contains(regex, case=False, na=False).sum()
        pct = round(count / total * 100, 2)
        rows.append({"Pattern": pattern_name, "Count": count, "Pct": pct})

    return pd.DataFrame(rows)


def show_examples(df, col, regex, n=3):
    """Print n example values matching a pattern -- for human judgment."""
    examples = (
        df[df[col].str.contains(regex, case=False, na=False)][col]
        .drop_duplicates()
        .head(n)
        .tolist()
    )
    for ex in examples:
        print(f"    -> {ex}")

---
## ③ Step 1 — 基礎 Pattern 統計（對應藍圖表 Step 1）

省下 Day1-3 重複貼三次程式碼的地方——一個 for 迴圈跑完 D1/D2/D3 的 artist 欄位 + song 欄位，總共 6 張報表。

In [4]:
print("[STEP 1] Basic Pattern Statistics（基礎 Pattern 統計）")

for dataset_name, cfg in DATASETS.items():
    df = cfg["df"]
    artist_col = cfg["artist_col"]
    song_col = cfg["song_col"]

    print(f"\n{'='*55}")
    print(f"  {dataset_name} -- artist column: '{artist_col}'")
    print(f"{'='*55}")
    artist_stats = run_pattern_scan(df, artist_col, PATTERNS_ARTIST)
    print(artist_stats.to_string(index=False))

    print(f"\n{'='*55}")
    print(f"  {dataset_name} -- song column: '{song_col}'")
    print(f"{'='*55}")
    song_stats = run_pattern_scan(df, song_col, PATTERNS_SONG)
    print(song_stats.to_string(index=False))

[STEP 1] Basic Pattern Statistics（基礎 Pattern 統計）

  D1 -- artist column: 'artist'


                  Pattern  Count  Pct
     Contains 'Featuring'  32274 9.78
Contains 'feat.' or 'ft.'    441 0.13
             Contains '&'  25155 7.62
             Contains ','   4206 1.27
    Contains '()' or '[]'   1324 0.40
             Contains '/'    766 0.23
    Contains 'x' (Collab)    844 0.26
    Contains '+' (Collab)    651 0.20

  D1 -- song column: 'song'


              Pattern  Count  Pct
        Contains '()'  22882 6.93
        Contains '[]'    112 0.03
Contains ' - ' (dash)    872 0.26
     Contains 'feat.'      0 0.00
  Contains 'Remaster'      0 0.00
      Contains 'Live'   1770 0.54
     Contains 'Remix'     94 0.03
Contains 'Radio Edit'      0 0.00

  D2 -- artist column: 'artist'
                  Pattern  Count  Pct
     Contains 'Featuring'   1658 4.03
Contains 'feat.' or 'ft.'     10 0.02
             Contains '&'   1786 4.34
             Contains ','    255 0.62
    Contains '()' or '[]'     63 0.15
             Contains '/'     40 0.10
    Contains 'x' (Collab)     40 0.10
    Contains '+' (Collab)     24 0.06

  D2 -- song column: 'track'


              Pattern  Count  Pct
        Contains '()'   2831 6.89
        Contains '[]'     92 0.22
Contains ' - ' (dash)   2857 6.95
     Contains 'feat.'    148 0.36
  Contains 'Remaster'    865 2.10
      Contains 'Live'    604 1.47
     Contains 'Remix'    159 0.39
Contains 'Radio Edit'     40 0.10

  D3 -- artist column: 'artist_name'
                  Pattern  Count  Pct
     Contains 'Featuring'      0 0.00
Contains 'feat.' or 'ft.'      2 0.01
             Contains '&'    844 2.97
             Contains ','    224 0.79
    Contains '()' or '[]'      3 0.01
             Contains '/'     45 0.16
    Contains 'x' (Collab)      9 0.03
    Contains '+' (Collab)     16 0.06

  D3 -- song column: 'track_name'
              Pattern  Count  Pct
        Contains '()'   1480 5.22
        Contains '[]'     24 0.08
Contains ' - ' (dash)      0 0.00
     Contains 'feat.'    357 1.26
  Contains 'Remaster'     24 0.08
      Contains 'Live'    234 0.82
     Contains 'Remix'     28 0.10
Contain

---
## ① CONFIG — Step 2-4 專用的分類規則（對應藍圖表 Step 2/3/4）

R 版本只對 **D1** 做這層深度分析，D2/D3 R 版本只寫了統計數字 + 手動判斷的文字結論，沒有寫這層 `case_when()` 分類程式碼。這裡照實移植：D1 做完整分類，D2/D3 沿用 R 的手動判斷結論（存進最後 Step 6 的 `CLEANING_RULES`）。

這裡的 CONFIG 定義的是「怎麼分類」的規則（哪些關鍵字算 Remix、哪些算合作標記…），跟 Step 1 的 `PATTERNS_ARTIST`/`PATTERNS_SONG` 是同一種東西，只是回答的問題更細——不只「有沒有這個符號」，還要「這個符號背後代表什麼意思」。

In [5]:
# ============================================================
# CONFIG -- classification rules for Step 2-4 (Deep Analysis)
# ============================================================

# CONFIG -- classification rules for symbols that need collab-marker context
COLLAB_MARKER_REGEX = r'featuring|feat\.|ft\.|with'

# CONFIG -- classification rules for parenthetical content (song titles)
SONG_PAREN_CATEGORIES = [
    ("Remix Version",         r'remix'),
    ("Performance Version",   r'^live$|live version|acoustic|unplugged'),
    ("Release Version Tag",   r'radio edit|album version|single version'),
    ("Remastered Version",    r'remaster'),
    ("Featuring Info",        r'feat\.|featuring'),
]
SONG_PAREN_DEFAULT = "Subtitle or Title Part (Keep)"

# CONFIG -- artist column uses a DIFFERENT, simpler rule (mirrors R exactly --
# do NOT reuse SONG_PAREN_CATEGORIES here, they answer a different question)
ARTIST_PAREN_CATEGORIES = [
    ("Collab-related (Safe to remove)", r'feat|duet|featuring'),
]
ARTIST_PAREN_DEFAULT = "Group Info/Metadata (Preserve)"

# CONFIG -- content after a " - " dash (song titles only, D1)
DASH_CATEGORIES = [
    ("Remix Version",         r'remix'),
    ("Remastered Version",    r'remaster'),
    ("Performance Version",   r'live|acoustic'),
    ("Release Version Tag",   r'radio edit|album version|single'),
]
DASH_DEFAULT = "Subtitle or Title Part (Keep)"

---
## ② Engine — Step 2-4 專用的分類 function（對應整體架構圖的②，通用、吃上面的 CONFIG 當參數，不判斷該不該清）

`classify_symbol_usage()` 跟 `classify_extracted_content()`／`classify_parenthetical_content()`——這三個函數本身完全通用，因為判斷規則都收進上面的 CONFIG 了，以後要套用到 D2/D3 也只要呼叫，不用重寫邏輯。這格只是**定義**，不會印出任何結果。

In [6]:
def classify_symbol_usage(df, col, symbol_regex):
    """Classify rows containing `symbol_regex` by collab-marker context + repeat count.
    Mirrors R's Analysis 1 (& symbol) / Analysis 3 (comma) case_when() logic."""
    data = df[df[col].str.contains(symbol_regex, na=False)][[col]].drop_duplicates().copy()
    data["has_collab_marker"] = data[col].str.contains(COLLAB_MARKER_REGEX, case=False, na=False)
    data["symbol_count"] = data[col].str.count(symbol_regex)

    def _type(row):
        if row["has_collab_marker"]:
            return "Has collab marker (Safe to clean)"
        elif row["symbol_count"] >= 2:
            return "Multiple (Complex)"
        else:
            return "Simple (Likely duo/group or part of name)"

    data["type"] = data.apply(_type, axis=1)
    total = len(data)
    summary = (data["type"].value_counts()
               .rename_axis("Type").reset_index(name="Count"))
    summary["Pct"] = round(summary["Count"] / total * 100, 2)
    return data, summary


def classify_extracted_content(df, col, contains_regex, extract_regex, categories, default_label):
    """Filter rows matching `contains_regex`, extract the substring matched by
    `extract_regex` (group 1), then classify it by `categories` (first match wins,
    else `default_label`). Generalizes R's Analysis-2-style case_when() blocks --
    used for both parentheses content and post-dash content."""
    mask = df[col].str.contains(contains_regex, na=False)
    data = df[mask][[col]].drop_duplicates().copy()
    data["extracted"] = data[col].str.extract(extract_regex)[0]

    def _type(text):
        if pd.isna(text):
            return default_label
        for label, regex in categories:
            if pd.Series([text]).str.contains(regex, case=False, regex=True).iloc[0]:
                return label
        return default_label

    data["type"] = data["extracted"].apply(_type)
    # 核心大機器內部的尾聲：
    total = len(data)                                          # 1. 先計算總共有幾個人有括號 (共 87 人)
    # 2. 叫電腦開始數人頭（每個標籤各有幾筆）
    summary = (data["type"].value_counts()
               .rename_axis("Type").reset_index(name="Count"))
    
    # 3. 自動計算百分比，並四捨五入到小數點第二位
    summary["Pct"] = round(summary["Count"] / total * 100, 2)
    # 4. 打包成漂亮的表格交出來
    return data, summary 


def classify_parenthetical_content(df, col, categories, default_label):
    """Thin wrapper: parentheses is the most common case of classify_extracted_content."""
    # 💡 就在這裡！它在 return 的時候，悄悄呼叫了底層無敵大機器！
    return classify_extracted_content(
        df, col, r'\(.*\)', r'\(([^)]+)\)', categories, default_label)

### ③ Step 2 — '&' / ',' 分類（對應藍圖表 Step 2，D1 only）

In [7]:
print("[STEP 2] '&' / ',' Usage Classification（& / 逗號分類，D1 only）")

print("\n=== D1 artist: '&' usage classification ===")
d1_amp_data, d1_amp_summary = classify_symbol_usage(d1_billboard, "artist", "&")
print(f"Total artists containing '&': {len(d1_amp_data):,}\n")
print(d1_amp_summary.to_string(index=False))

print("\n=== D1 artist: ',' usage classification ===")
d1_comma_data, d1_comma_summary = classify_symbol_usage(d1_billboard, "artist", ",")
print(f"Total artists containing ',': {len(d1_comma_data):,}\n")
print(d1_comma_summary.to_string(index=False))

[STEP 2] '&' / ',' Usage Classification（& / 逗號分類，D1 only）

=== D1 artist: '&' usage classification ===
Total artists containing '&': 1,485

                                     Type  Count   Pct
Simple (Likely duo/group or part of name)    866 58.32
        Has collab marker (Safe to clean)    612 41.21
                       Multiple (Complex)      7  0.47

=== D1 artist: ',' usage classification ===


Total artists containing ',': 290

                                     Type  Count   Pct
        Has collab marker (Safe to clean)    148 51.03
Simple (Likely duo/group or part of name)    108 37.24
                       Multiple (Complex)     34 11.72


### ③ Step 3 — 括號內容分類（對應藍圖表 Step 3）

⚠️ artist 欄位跟 song 欄位用**不同的 CONFIG**（`ARTIST_PAREN_CATEGORIES` vs `SONG_PAREN_CATEGORIES`）——同一個 function，因為判斷規則不同，不能共用。

📌 **2026/07/20 核對 R 原始結果時確認的方法論決定**：`classify_extracted_content()` 在分類前一律先 `.drop_duplicates()`，也就是**只算「不重複的藝人/歌名字串」，不是「原始上榜列數」**。R 原始版本這裡沒有去重（`nrow(paren_data)` 算出的是 1271，包含同一位藝人上榜多週被重複計算），跟 Python 這裡算出的「87 位不重複藝人」對不上——但 R 自己在 `&` 符號分析（Step 2）反而是有去重的（跟 Python 對得上）。**確認過：R 原始版本兩個分析用的方法本來就不一致，不是 Python 轉譯出錯**。這裡刻意統一維持「不重複字串」這個口徑（跟 Step 2 一致），是明確的人工決定，不是疏漏。

In [8]:
print("[STEP 3] Parenthetical Content Classification（括號內容分類，D1 only）")

print("\n=== D1 artist: parentheses content classification ===")
d1_artist_paren_data, d1_artist_paren_summary = classify_parenthetical_content(
    d1_billboard, "artist", ARTIST_PAREN_CATEGORIES, ARTIST_PAREN_DEFAULT)
print(f"Total artists containing parentheses: {len(d1_artist_paren_data):,}\n")
print(d1_artist_paren_summary.to_string(index=False))

print("\n=== D1 song: parentheses content classification ===")
d1_song_paren_data, d1_song_paren_summary = classify_parenthetical_content(
    d1_billboard, "song", SONG_PAREN_CATEGORIES, SONG_PAREN_DEFAULT)
print(f"Total songs containing parentheses: {len(d1_song_paren_data):,}\n")
print(d1_song_paren_summary.to_string(index=False))

[STEP 3] Parenthetical Content Classification（括號內容分類，D1 only）

=== D1 artist: parentheses content classification ===
Total artists containing parentheses: 87

                           Type  Count   Pct
Collab-related (Safe to remove)     45 51.72
 Group Info/Metadata (Preserve)     42 48.28

=== D1 song: parentheses content classification ===


Total songs containing parentheses: 1,995

                         Type  Count   Pct
Subtitle or Title Part (Keep)   1983 99.40
          Performance Version      6  0.30
                Remix Version      5  0.25
               Featuring Info      1  0.05


### ③ Step 4 — 破折號後內容分類（對應藍圖表 Step 4，D1 song only — R 版本只有 D1 song 有這個分析）

In [9]:
print("[STEP 4] Post-Dash Content Classification（破折號後內容分類，D1 song only）")

d1_dash_data, d1_dash_summary = classify_extracted_content(
    d1_billboard, "song", r' - ', r' - (.*)$', DASH_CATEGORIES, DASH_DEFAULT)
print(f"Total songs containing ' - ': {len(d1_dash_data):,}\n")
print(d1_dash_summary.to_string(index=False))

[STEP 4] Post-Dash Content Classification（破折號後內容分類，D1 song only）
Total songs containing ' - ': 96

                         Type  Count   Pct
Subtitle or Title Part (Keep)     95 98.96
                Remix Version      1  1.04


---
## ③ Step 5 — 跨資料集比較（對應藍圖表 Step 5）

D1 vs D2 vs D3 — artist patterns

In [10]:
print("[STEP 5] Cross-Dataset Comparison（跨資料集比較）")

three_way = None
for dataset_name, cfg in DATASETS.items():
    stats = run_pattern_scan(cfg["df"], cfg["artist_col"], PATTERNS_ARTIST)
    stats = stats.rename(columns={"Pct": f"{dataset_name} %"})[["Pattern", f"{dataset_name} %"]]
    three_way = stats if three_way is None else three_way.merge(stats, on="Pattern")

print("\nD1 vs D2 vs D3 -- Artist Pattern Comparison (%)")
print("=" * 60)
print(three_way.to_string(index=False))

[STEP 5] Cross-Dataset Comparison（跨資料集比較）



D1 vs D2 vs D3 -- Artist Pattern Comparison (%)
                  Pattern  D1 %  D2 %  D3 %
     Contains 'Featuring'  9.78  4.03  0.00
Contains 'feat.' or 'ft.'  0.13  0.02  0.01
             Contains '&'  7.62  4.34  2.97
             Contains ','  1.27  0.62  0.79
    Contains '()' or '[]'  0.40  0.15  0.01
             Contains '/'  0.23  0.10  0.16
    Contains 'x' (Collab)  0.26  0.10  0.03
    Contains '+' (Collab)  0.20  0.06  0.06


---
## ④ Step 6 — `CLEANING_RULES`（對應藍圖表 Step 6，最終交付物，hand-authored）

**這份不是 engine 算出來的，是人工判斷的結果**——把 Step 1-5 所有統計 + R 版本原本用文字寫的「Cleaning Strategy」段落，轉成結構化的 config。03 的 `clean_column(series, rules)` 會直接吃這份 dict 跑，不用再為每個資料集寫一次清理邏輯。

In [11]:
print("[STEP 6] CLEANING_RULES（最終交付物）\n")

CLEANING_RULES = {
    "D1": {
        "artist": {
            "remove_after_marker": [r"featuring", r"feat\.", r"ft\.", r"with"],
            "remove_parens_if_contains": [r"feat", r"duet", r"featuring"],
            "preserve": ["&", ",", "brackets not matching collab markers"],
        },
        "song": {
            "remove_version_tags_in_parens": [r"remix", r"live version", r"radio edit",
                                               r"album version", r"single version", r"remaster"],
            "remove_version_tags_after_dash": [r"remix", r"remaster", r"live", r"acoustic",
                                                r"radio edit", r"album version", r"single"],
            "remove_brackets_entirely": True,
            "preserve": ["standalone parens/dash treated as title subtitle",
                         "'live'/'part'/'take' as lyric words -- false positive risk, do not blanket-remove"],
        },
    },
    "D2": {
        "artist": {
            "remove_after_marker": [r"featuring", r"feat\.", r"ft\.", r"with"],
            "preserve": ["&", "/", "+", "x", "brackets", "commas"],
        },
        "track": {
            "remove_brackets_entirely": True,
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"live", r"radio edit", r"feat\."],
            "remove_version_tags_after_dash": [r"remaster", r"live", r"remix", r"radio edit"],
            "note": "D2 dash usage is ACTIVE version-tagging (unlike D1's mostly-subtitle dash) -- clean more aggressively",
            "preserve": ["'Part'/'Pt.' after dash -- track numbering, not a version tag"],
        },
    },
    "D3": {
        "artist_name": {
            "action": "trim_whitespace_only",
            "note": "0% Featuring, <3% any symbol -- essentially clean already, no removal needed",
        },
        "track_name": {
            "remove_brackets_entirely": True,
            "remove_version_tags_in_parens": [r"remix", r"remaster", r"feat\."],
            "caution": ["'live' -- most occurrences inside parens are version tags, but 'live' "
                        "outside parens is frequently a lyric word ('as long as i live'); "
                        "only remove '(live)'-style explicit tags"],
        },
    },
}

print("CLEANING_RULES defined for D1 / D2 / D3 -- ready for 03_data_wrangling")
for ds, cols in CLEANING_RULES.items():
    print(f"  {ds}: {list(cols.keys())}")

[STEP 6] CLEANING_RULES（最終交付物）

CLEANING_RULES defined for D1 / D2 / D3 -- ready for 03_data_wrangling
  D1: ['artist', 'song']
  D2: ['artist', 'track']
  D3: ['artist_name', 'track_name']
